# 05 - RQ3: Causal Uplift & The Sleeping Dog Blacklist

| Component | Method | Key correction |
|---|---|---|
| Balancing | **IPTW** (stabilised ATE weights) | Eliminates endogeneity in T/C assignment |
| CATE | T-Learner (two IPTW-weighted XGBoost) | Sample weights simulate RCT |
| Validation | Qini curve + AUUC | Formal uplift performance metric |
| ROI | Monte Carlo (10k iters, 95% CI) | Point estimates replaced with intervals |
| Segments | Four-quadrant framework | Persuadable / Sure-Thing / Sleeping Dog / Lost Cause |
| Ablation | RFM-only vs RFM + BG/NBD | Quantifies RQ2 -> RQ3 value chain |

In [1]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import xgboost as xgb
import scipy.stats as stats
warnings.filterwarnings("ignore")

DATA_DIR   = "../outputs/data/"
CHARTS_DIR = "../outputs/charts/"
MODELS_DIR = "../outputs/models/"
os.makedirs(MODELS_DIR, exist_ok=True)

PALETTE = {
    "primary": "#2E75B6", "accent": "#E63946",
    "positive": "#2A9D8F", "highlight": "#F4A261",
    "neutral": "#8D99AE", "bg": "#FAFAFA", "grid": "#E8E8E8",
}
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.facecolor": PALETTE["bg"], "figure.facecolor": PALETTE["bg"],
    "axes.grid": True, "grid.color": PALETTE["grid"],
    "grid.linestyle": "--", "font.size": 11,
    "axes.titlesize": 13, "axes.titleweight": "bold",
})

rq2 = pd.read_csv(f"{DATA_DIR}04_rq2_customer_health.csv")
txn = pd.read_csv(f"{DATA_DIR}01_transactions_clean.csv",
                  parse_dates=["InvoiceDate"])

print(f"RQ2 customers : {len(rq2):,}")
print(f"Transactions  : {len(txn):,}")
print(f"RQ2 columns   : {rq2.columns.tolist()}")

RQ2 customers : 5,852
Transactions  : 820,221
RQ2 columns   : ['CustomerID', 'LastPurchase', 'FirstPurchase', 'NumTransactions', 'TotalSpend', 'TotalItems', 'UniqueProducts', 'UniqueCategories', 'PrimaryCountry', 'Recency', 'CustomerLifespan', 'AOV', 'PurchaseFreqPerMonth', 'IsUK', 'R_Score', 'F_Score', 'M_Score', 'RFM_Score', 'CustomerSegment', 'behavioral_entropy', 'p_alive', 'predicted_clv_12m', 'mu', 'alpha', 'beta']


## Step 1 - Treatment and outcome definition

In [2]:
print("=" * 60)
print("  STEP 1: TREATMENT & OUTCOME DEFINITION  (FIXED)")
print("=" * 60)
print()
print("  DESIGN CHANGE: time-split outcome removes endogeneity.")
print("  Observation window : first 18 months (Dec 2009 – May 2011)")
print("  Outcome window     : last 6 months  (Jun 2011 – Dec 2011)")
print("  Treatment          : customers who went 60-180d dormant by end of obs window")
print("  Outcome            : purchased in the outcome window (binary)")
print("  This is independent of the recency-derived R_Score - no leakage.")

# ── Time split ────────────────────────────────────────────────────────────────
OBS_END   = pd.Timestamp("2011-05-31")   # end of observation window
OUT_START = pd.Timestamp("2011-06-01")   # start of outcome window

# Load raw transactions to build time-split outcome
txn = pd.read_csv(f"{DATA_DIR}01_transactions_clean.csv", parse_dates=["InvoiceDate"])
txn_obs  = txn[txn["InvoiceDate"] <= OBS_END]
txn_out  = txn[txn["InvoiceDate"] >  OBS_END]

# Customer last purchase in obs window
last_obs = (
    txn_obs.groupby("CustomerID")["InvoiceDate"].max()
    .reset_index()
    .rename(columns={"InvoiceDate": "LastPurchaseObs"})
)

# Who purchased in outcome window?
out_buyers = set(txn_out["CustomerID"].unique())

model_df = rq2.merge(last_obs, on="CustomerID", how="left")
model_df["LastPurchaseObs"] = model_df["LastPurchaseObs"].fillna(
    pd.Timestamp("2009-12-01")
)

# Recency at end of obs window
model_df["RecencyAtObsEnd"] = (
    OBS_END - model_df["LastPurchaseObs"]
).dt.days

# Treatment: dormant at end of obs window (60–180 days since last purchase)
TREATMENT_LOW  = 60
TREATMENT_HIGH = 180
model_df["treatment"] = (
    (model_df["RecencyAtObsEnd"] >= TREATMENT_LOW) &
    (model_df["RecencyAtObsEnd"] <= TREATMENT_HIGH)
).astype(int)

# Outcome: did they purchase in outcome window?  FULLY INDEPENDENT of treatment definition.
model_df["outcome"] = model_df["CustomerID"].isin(out_buyers).astype(int)

n_treat = model_df["treatment"].sum()
n_ctrl  = (model_df["treatment"] == 0).sum()
n_out   = model_df["outcome"].sum()
n_total = len(model_df)

print(f"\n  Total population   : {n_total:,}")
print(f"  Treatment (1)      : {n_treat:,}  ({n_treat/n_total:.1%})  [Recency 60-180d at obs end]")
print(f"  Control   (0)      : {n_ctrl:,}  ({n_ctrl/n_total:.1%})")
print(f"  Outcome = 1        : {n_out:,}  ({n_out/n_total:.1%})")
out_t = model_df.loc[model_df["treatment"]==1, "outcome"].mean()
out_c = model_df.loc[model_df["treatment"]==0, "outcome"].mean()
print(f"\n  Outcome rate (treat): {out_t:.1%}  <- must be > 0 for T-Learner to work")
print(f"  Outcome rate (ctrl) : {out_c:.1%}")
if out_t == 0:
    print("  WARNING: outcome_rate(treat) = 0 - check time split or treatment window")
else:
    print("  OK: Both groups have positive outcome rates - T-Learner can estimate CATE.")


  STEP 1: TREATMENT & OUTCOME DEFINITION  (FIXED)

  DESIGN CHANGE: time-split outcome removes endogeneity.
  Observation window : first 18 months (Dec 2009 – May 2011)
  Outcome window     : last 6 months  (Jun 2011 – Dec 2011)
  Treatment          : customers who went 60-180d dormant by end of obs window
  Outcome            : purchased in the outcome window (binary)
  This is independent of the recency-derived R_Score - no leakage.

  Total population   : 5,852
  Treatment (1)      : 1,192  (20.4%)  [Recency 60-180d at obs end]
  Control   (0)      : 4,660  (79.6%)
  Outcome = 1        : 3,575  (61.1%)

  Outcome rate (treat): 58.4%  <- must be > 0 for T-Learner to work
  Outcome rate (ctrl) : 61.8%
  OK: Both groups have positive outcome rates - T-Learner can estimate CATE.


## Step 2 - Feature preparation and imputation

In [3]:
print("=" * 60)
print("  STEP 2: FEATURE PREPARATION + IMPUTATION")
print("=" * 60)

PALIVE_MEDIAN = rq2["p_alive"].median()
CLV_MEDIAN    = rq2["predicted_clv_12m"].median()

model_df["p_alive_imputed"] = model_df["p_alive"].fillna(PALIVE_MEDIAN)
model_df["clv_imputed"]     = model_df["predicted_clv_12m"].fillna(0.0)

# ── Hawkes imputation (median fill - ~44% of customers not fitted) ────────
MU_MEDIAN    = rq2["mu"].median()
ALPHA_MEDIAN = rq2["alpha"].median()
model_df["mu_imputed"]    = model_df["mu"].fillna(MU_MEDIAN)
model_df["alpha_imputed"] = model_df["alpha"].fillna(ALPHA_MEDIAN)

print(f"  p_alive median imputed : {PALIVE_MEDIAN:.3f}")
print(f"  CLV nulls filled -> 0  : {model_df['predicted_clv_12m'].isna().sum():,}")
print(f"  mu median imputed      : {MU_MEDIAN:.4f}")
print(f"  alpha median imputed   : {ALPHA_MEDIAN:.4f}")
print(f"  Hawkes nulls (mu)      : {model_df['mu'].isna().sum():,}")

# ── Feature columns ───────────────────────────────────────────────────────────
# NOT included: Recency (defines treatment), R_Score (defines outcome)
FEATURE_COLS = [
    # Core RFM - complete for all customers
    "NumTransactions",
    "TotalSpend",
    "AOV",
    "CustomerLifespan",
    "PurchaseFreqPerMonth",
    "behavioral_entropy",
    "IsUK",
    "F_Score",
    "M_Score",
    # BG/NBD outputs - imputed (defensible single-value fill)
    "p_alive_imputed",
    "clv_imputed",
    # Hawkes temporal dynamics - imputed at median for non-fitted customers
    "mu_imputed",
    "alpha_imputed",
]

null_check = model_df[FEATURE_COLS].isna().sum()
if null_check.sum() == 0:
    print(f"  Zero nulls in {len(FEATURE_COLS)}-column feature matrix.")
else:
    print(f"  Remaining nulls: {null_check[null_check > 0].to_dict()}")

X = model_df[FEATURE_COLS].values
y = model_df["outcome"].values
t = model_df["treatment"].values

  STEP 2: FEATURE PREPARATION + IMPUTATION
  p_alive median imputed : 0.920
  CLV nulls filled -> 0  : 3,433
  mu median imputed      : 0.0159
  alpha median imputed   : 0.0000
  Hawkes nulls (mu)      : 2,563
  Zero nulls in 13-column feature matrix.


## Step 3 - IPTW: Inverse Probability of Treatment Weighting

Without IPTW the T-Learner is endogenous: frequent buyers have lower recency
by definition, so the model just learns "frequent buyers buy again" - not the
causal effect of an intervention.

IPTW re-weights each customer by `1/P(T=1|X)` for treated and
`1/P(T=0|X)` for control, statistically balancing covariates and
simulating a randomised controlled trial on observational data.

In [4]:
print("=" * 60)
print("  STEP 3: IPTW - INVERSE PROBABILITY OF TREATMENT WEIGHTING")
print("=" * 60)
print()
print("  Without IPTW the T-Learner just learns 'frequent buyers buy again'.")
print("  IPTW re-weights each customer by the inverse of their probability")
print("  of being in the treatment group, simulating a randomised trial.")
print()

# ── Propensity score model ────────────────────────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

ps_model = LogisticRegression(
    C=1.0, max_iter=1000, random_state=42, solver="lbfgs"
)
ps_model.fit(X_scaled, t)

propensity = ps_model.predict_proba(X_scaled)[:, 1]
model_df["propensity_score"] = propensity

# ── Clip extreme propensity scores to [0.01, 0.99] ───────────────────────────
# Trimming prevents extreme weights from dominating (Crump et al. 2009)
propensity_clipped = np.clip(propensity, 0.01, 0.99)

# ── IPTW weights ──────────────────────────────────────────────────────────────
# ATE weights: w = T/ps + (1-T)/(1-ps)
weights_ate = np.where(
    t == 1,
    1.0 / propensity_clipped,
    1.0 / (1.0 - propensity_clipped),
)
# Stabilised ATE weights (Smith & Todd 2005) - reduces variance
p_t = t.mean()
weights_stable = np.where(
    t == 1,
    p_t / propensity_clipped,
    (1 - p_t) / (1.0 - propensity_clipped),
)
model_df["iptw_weight"] = weights_stable

print(f"  Propensity score range : [{propensity.min():.3f}, {propensity.max():.3f}]")
print(f"  Propensity mean (treat): "
      f"{propensity[t==1].mean():.3f}  (control: {propensity[t==0].mean():.3f})")
print(f"  Stabilised IPTW weights: "
      f"mean={weights_stable.mean():.2f}  max={weights_stable.max():.1f}")

# ── Covariate balance check ───────────────────────────────────────────────────
balance_rows = []
for col in FEATURE_COLS:
    raw = model_df.groupby("treatment")[col].mean()
    smd_raw = (raw[1] - raw[0]) / model_df[col].std()
    wt_mean_t = np.average(
        model_df.loc[model_df["treatment"]==1, col],
        weights=model_df.loc[model_df["treatment"]==1, "iptw_weight"],
    )
    wt_mean_c = np.average(
        model_df.loc[model_df["treatment"]==0, col],
        weights=model_df.loc[model_df["treatment"]==0, "iptw_weight"],
    )
    smd_wt = (wt_mean_t - wt_mean_c) / model_df[col].std()
    balance_rows.append({
        "feature": col,
        "mean_treat_raw": raw[1],
        "mean_ctrl_raw":  raw[0],
        "smd_raw":        round(smd_raw, 3),
        "mean_treat_wt":  round(wt_mean_t, 3),
        "mean_ctrl_wt":   round(wt_mean_c, 3),
        "smd_weighted":   round(smd_wt, 3),
        "balanced_05":    abs(smd_wt) < 0.10,
    })

balance_df = pd.DataFrame(balance_rows)
print()
print("  Covariate balance (|SMD| < 0.10 = well-balanced):")
print(balance_df[["feature", "smd_raw", "smd_weighted", "balanced_05"]].to_string(index=False))

n_balanced = balance_df["balanced_05"].sum()
print(f"\n  {n_balanced}/{len(balance_df)} features balanced after IPTW.")

balance_df.to_csv(f"{DATA_DIR}05_iptw_balance.csv", index=False)
model_df[["CustomerID", "propensity_score", "iptw_weight"]].to_csv(
    f"{DATA_DIR}05_propensity_scores.csv", index=False
)
print("  CSVs: 05_iptw_balance.csv  05_propensity_scores.csv")

  STEP 3: IPTW - INVERSE PROBABILITY OF TREATMENT WEIGHTING

  Without IPTW the T-Learner just learns 'frequent buyers buy again'.
  IPTW re-weights each customer by the inverse of their probability
  of being in the treatment group, simulating a randomised trial.

  Propensity score range : [0.000, 0.514]
  Propensity mean (treat): 0.254  (control: 0.191)
  Stabilised IPTW weights: mean=1.00  max=20.4

  Covariate balance (|SMD| < 0.10 = well-balanced):
             feature  smd_raw  smd_weighted  balanced_05
     NumTransactions   -0.031         0.171        False
          TotalSpend   -0.044         0.025         True
                 AOV   -0.019        -0.018         True
    CustomerLifespan    0.419        -0.030         True
PurchaseFreqPerMonth   -0.398         0.309        False
  behavioral_entropy    0.105        -0.023         True
                IsUK   -0.028        -0.004         True
             F_Score    0.361        -0.016         True
             M_Score    0.29

## Step 4 - Ablation study: RFM baseline vs BG/NBD-enriched

In [5]:
print("=" * 60)
print("  STEP 4: ABLATION STUDY - RFM vs RFM + BG/NBD")
print("=" * 60)
print()
print("  Quantifies: does adding p_alive + CLV actually improve causal")
print("  estimation? If yes, RQ2 modelling is justified by RQ3 outcomes.")

BASE_FEATURES = [
    "NumTransactions", "TotalSpend", "AOV",
    "CustomerLifespan", "behavioral_entropy", "IsUK",
    "F_Score", "M_Score",
]
FULL_FEATURES = BASE_FEATURES + ["p_alive_imputed", "clv_imputed"]

XGB_PARAMS = dict(
    n_estimators=300, learning_rate=0.05, max_depth=4,
    subsample=0.8, colsample_bytree=0.8,
    min_child_weight=10, reg_lambda=2.0, random_state=42,
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Baseline
auc_base = cross_val_score(
    xgb.XGBClassifier(**XGB_PARAMS),
    model_df[BASE_FEATURES].values, y,
    cv=cv, scoring="roc_auc",
).mean()

# Enriched
auc_full = cross_val_score(
    xgb.XGBClassifier(**XGB_PARAMS),
    model_df[FULL_FEATURES].values, y,
    cv=cv, scoring="roc_auc",
).mean()

delta    = auc_full - auc_base
lift_pct = delta / auc_base * 100 if auc_base > 0 else 0.0

print(f"  Baseline AUC (RFM only)   : {auc_base:.4f}")
print(f"  Enriched AUC (+ BG/NBD)   : {auc_full:.4f}")
print(f"  Delta AUC                  : {delta:+.4f}  ({lift_pct:+.1f}%)")
if delta > 0:
    print("  BG/NBD features IMPROVE causal model performance.")
    print(f"  RQ2 -> RQ3 value chain confirmed (+{delta:.4f} AUC).")
else:
    print("  BG/NBD features did not improve - check imputation strategy.")

  STEP 4: ABLATION STUDY - RFM vs RFM + BG/NBD

  Quantifies: does adding p_alive + CLV actually improve causal
  estimation? If yes, RQ2 modelling is justified by RQ3 outcomes.
  Baseline AUC (RFM only)   : 0.8198
  Enriched AUC (+ BG/NBD)   : 0.8868
  Delta AUC                  : +0.0671  (+8.2%)
  BG/NBD features IMPROVE causal model performance.
  RQ2 -> RQ3 value chain confirmed (+0.0671 AUC).


## Step 5 - T-Learner CATE estimation (IPTW-weighted)

In [6]:
print("=" * 60)
print("  STEP 5: T-LEARNER CATE ESTIMATION (IPTW-WEIGHTED)")
print("=" * 60)
print()
print("  CATE(x) = E[Y|X=x, T=1] - E[Y|X=x, T=0]")
print("  Sample weights = stabilised IPTW to simulate RCT on observational data.")

mask_t = model_df["treatment"] == 1
mask_c = model_df["treatment"] == 0

X_t = model_df.loc[mask_t, FEATURE_COLS].values
y_t = model_df.loc[mask_t, "outcome"].values
w_t = model_df.loc[mask_t, "iptw_weight"].values

X_c = model_df.loc[mask_c, FEATURE_COLS].values
y_c = model_df.loc[mask_c, "outcome"].values
w_c = model_df.loc[mask_c, "iptw_weight"].values

print(f"  Treatment group : {len(X_t):,}")
print(f"  Control group   : {len(X_c):,}")

model_t = xgb.XGBClassifier(**XGB_PARAMS)
model_c = xgb.XGBClassifier(**XGB_PARAMS)
model_t.fit(X_t, y_t, sample_weight=w_t)
model_c.fit(X_c, y_c, sample_weight=w_c)

X_full = model_df[FEATURE_COLS].values
p1     = model_t.predict_proba(X_full)[:, 1]
p0     = model_c.predict_proba(X_full)[:, 1]
cate   = p1 - p0

model_df["CATE"]     = cate
model_df["p1_score"] = p1
model_df["p0_score"] = p0

auc_t_cv = cross_val_score(model_t, X_t, y_t, cv=cv, scoring="roc_auc").mean()
auc_c_cv = cross_val_score(model_c, X_c, y_c, cv=cv, scoring="roc_auc").mean()

print(f"\n  Treatment model CV AUC : {auc_t_cv:.3f}")
print(f"  Control   model CV AUC : {auc_c_cv:.3f}")
print(f"  CATE mean : {cate.mean():.4f}  std: {cate.std():.4f}")
print(f"  CATE range: [{cate.min():.3f}, {cate.max():.3f}]")

  STEP 5: T-LEARNER CATE ESTIMATION (IPTW-WEIGHTED)

  CATE(x) = E[Y|X=x, T=1] - E[Y|X=x, T=0]
  Sample weights = stabilised IPTW to simulate RCT on observational data.
  Treatment group : 1,192
  Control group   : 4,660

  Treatment model CV AUC : 0.964
  Control   model CV AUC : 0.885
  CATE mean : -0.1844  std: 0.2840
  CATE range: [-0.942, 0.933]


## Step 6 - Four-quadrant segment assignment

In [7]:
def assign_segment(cate_val):
    if cate_val > 0.05:
        return "Persuadable"
    if cate_val > 0.0:
        return "Sure-Thing"
    if cate_val >= -0.05:
        return "Sleeping Dog"
    return "Lost Cause"


model_df["UpliftSegment"] = model_df["CATE"].apply(assign_segment)

seg_counts = model_df["UpliftSegment"].value_counts()
seg_spend  = model_df.groupby("UpliftSegment")["TotalSpend"].sum()
seg_cate   = model_df.groupby("UpliftSegment")["CATE"].mean()

print("Uplift segment distribution:")
for seg in ["Persuadable", "Sure-Thing", "Sleeping Dog", "Lost Cause"]:
    n   = seg_counts.get(seg, 0)
    mc  = seg_cate.get(seg, 0.0)
    sp  = seg_spend.get(seg, 0.0)
    print(f"  {seg:<16}  n={n:>4}  ({n/len(model_df):.1%})  "
          f"mean CATE={mc:+.3f}  spend=GBP{sp:,.0f}")

sleeping_dogs = model_df[model_df["UpliftSegment"] == "Sleeping Dog"].copy()
sd_n     = len(sleeping_dogs)
sd_spend = sleeping_dogs["TotalSpend"].sum()
print()
print(f"  SLEEPING DOG INSIGHT:")
print(f"  {sd_n} customers  |  GBP{sd_spend:,.0f} historical spend")
print("  Your CRM suppression list should come from your BEST customers.")
print("  These accounts return on their own schedule - contact disrupts it.")

Uplift segment distribution:
  Persuadable       n= 708  (12.1%)  mean CATE=+0.280  spend=GBP1,129,280
  Sure-Thing        n= 290  (5.0%)  mean CATE=+0.019  spend=GBP572,831
  Sleeping Dog      n=1380  (23.6%)  mean CATE=-0.017  spend=GBP4,883,021
  Lost Cause        n=3474  (59.4%)  mean CATE=-0.362  spend=GBP10,844,276

  SLEEPING DOG INSIGHT:
  1380 customers  |  GBP4,883,021 historical spend
  Your CRM suppression list should come from your BEST customers.
  These accounts return on their own schedule - contact disrupts it.


## Step 7 - Counterfactual CLV + Monte Carlo ROI (95% CI)

In [8]:
print("=" * 60)
print("  STEP 6: COUNTERFACTUAL CLV + MONTE CARLO ROI (95% CI)")
print("=" * 60)

HORIZON = 90

model_df["expected_orders"] = (
    model_df["NumTransactions"]
    / model_df["CustomerLifespan"].clip(lower=1)
    * HORIZON
)
cap_95 = model_df["expected_orders"].quantile(0.95)
model_df["expected_orders_capped"] = model_df["expected_orders"].clip(upper=cap_95)
model_df["CounterfactualCLV"] = np.where(
    model_df["CATE"] > 0,
    model_df["CATE"] * model_df["AOV"] * model_df["expected_orders_capped"],
    0.0,
)

priority = (
    model_df[model_df["UpliftSegment"] == "Persuadable"]
    .sort_values(["CounterfactualCLV", "p_alive_imputed"], ascending=False)
    .head(50)[[
        "CustomerID", "CATE", "CounterfactualCLV",
        "p_alive_imputed", "NumTransactions", "TotalSpend", "AOV",
    ]]
)
top50_clv = priority["CounterfactualCLV"].sum()
breakeven = top50_clv / max(len(priority), 1)

print(f"  Top-50 Persuadable CLV (capped) : GBP{top50_clv:,.0f}")
print(f"  Breakeven cost per contact       : GBP{breakeven:,.0f}")

# ── Monte Carlo with bootstrap 95% CI ────────────────────────────────────────
# Point estimate alone is fragile - present GBP X +/- Y (95% CI)
N_ITER     = 10_000
COST_RANGE = np.arange(2, 16, 1)
N_CONTACTS = 50
np.random.seed(42)

roi_records = []
for cost_per_contact in COST_RANGE:
    campaign_cost = cost_per_contact * N_CONTACTS
    # Sample CLV with 20% coefficient of variation (conservative uncertainty)
    clv_samples   = top50_clv * np.random.normal(1, 0.20, N_ITER).clip(0)
    roi_vals      = (clv_samples - campaign_cost) / max(campaign_cost, 1)
    roi_records.append({
        "cost_per_contact": cost_per_contact,
        "campaign_cost":    campaign_cost,
        "mean_roi":         roi_vals.mean(),
        "roi_p5":           np.percentile(roi_vals, 5),
        "roi_p95":          np.percentile(roi_vals, 95),
        "p_profitable":     (roi_vals > 0).mean(),
        "expected_return":  top50_clv,
    })

roi_df = pd.DataFrame(roi_records)
mid_row = roi_df.loc[roi_df["cost_per_contact"] == 7].iloc[0]

print(f"\n  At GBP7/contact (n=50 customers):")
print(f"    Expected ROI    : {mid_row['mean_roi']:.0%}")
print(f"    95% CI          : [{mid_row['roi_p5']:.0%}, {mid_row['roi_p95']:.0%}]")
print(f"    P(profitable)   : {mid_row['p_profitable']:.1%}")

roi_df.to_csv(f"{DATA_DIR}05_monte_carlo_roi.csv", index=False)
priority.to_csv(f"{DATA_DIR}05_intervention_priority.csv", index=False)
sleeping_dogs[["CustomerID","CATE","TotalSpend","NumTransactions","AOV"]].to_csv(
    f"{DATA_DIR}05_sleeping_dog_blacklist.csv", index=False
)
print("  CSVs: 05_monte_carlo_roi / intervention_priority / sleeping_dog_blacklist")

  STEP 6: COUNTERFACTUAL CLV + MONTE CARLO ROI (95% CI)
  Top-50 Persuadable CLV (capped) : GBP85,855
  Breakeven cost per contact       : GBP1,717

  At GBP7/contact (n=50 customers):
    Expected ROI    : 24439%
    95% CI          : [16235%, 32530%]
    P(profitable)   : 100.0%
  CSVs: 05_monte_carlo_roi / intervention_priority / sleeping_dog_blacklist


## Step 8 - Qini curve (formal uplift validation)

In [9]:
def compute_qini_curve(df):
    df_s = df.sort_values("CATE", ascending=False).copy()
    n    = len(df_s)
    n_t  = int(df_s["treatment"].sum())
    n_c  = n - n_t
    rows, cum_t_out, cum_c_out = [], 0, 0
    for _, row in df_s.iterrows():
        if row["treatment"] == 1:
            cum_t_out += row["outcome"]
        else:
            cum_c_out += row["outcome"]
        qini = (cum_t_out / n_t - cum_c_out / n_c) if (n_t > 0 and n_c > 0) else 0
        rows.append({"pct_contacted": (len(rows) + 1) / n, "qini_cate": qini})
    qini_df = pd.DataFrame(rows)
    final_q = qini_df["qini_cate"].iloc[-1]
    qini_df["qini_random"] = np.linspace(0, final_q, len(qini_df))
    return qini_df


qini_df = compute_qini_curve(model_df)
auuc = (qini_df["qini_cate"] - qini_df["qini_random"]).mean()

print(f"  AUUC (area between CATE curve and random) : {auuc:.4f}")
print(f"  {'Positive AUUC - CATE ranking outperforms random.' if auuc > 0 else 'Negative AUUC - check features.'}")

qini_df.to_csv(f"{DATA_DIR}05_qini_curve.csv", index=False)
print("  CSV: 05_qini_curve.csv")

  AUUC (area between CATE curve and random) : 0.1511
  Positive AUUC - CATE ranking outperforms random.
  CSV: 05_qini_curve.csv


## Step 9 - Feature importance

In [10]:
# Feature importance across both T-Learner models
imp_t = pd.Series(model_t.feature_importances_,
                  index=FEATURE_COLS, name="treatment_model")
imp_c = pd.Series(model_c.feature_importances_,
                  index=FEATURE_COLS, name="control_model")
imp_df = pd.DataFrame({"treatment": imp_t, "control": imp_c})
imp_df["mean_importance"] = imp_df.mean(axis=1)
imp_df = imp_df.sort_values("mean_importance", ascending=False)

print("Feature importances (mean across T and C models):")
for feat, row in imp_df.iterrows():
    bar = "X" * int(row["mean_importance"] * 60)
    print(f"  {feat:<25} {row['mean_importance']:.3f}  {bar}")

top_feat = imp_df.index[0]
top_imp  = imp_df["mean_importance"].iloc[0]
if top_imp < 0.25:
    print("  No single feature dominates (<25%) - genuine multivariate learning.")
else:
    print(f"  WARNING: {top_feat} dominates at {top_imp:.1%} - check for leakage.")

imp_df.reset_index().rename(columns={"index":"feature"}).to_csv(
    f"{DATA_DIR}05_feature_importance.csv", index=False
)
print("  CSV: 05_feature_importance.csv")

Feature importances (mean across T and C models):
  CustomerLifespan          0.251  XXXXXXXXXXXXXXX
  NumTransactions           0.128  XXXXXXX
  F_Score                   0.104  XXXXXX
  PurchaseFreqPerMonth      0.099  XXXXX
  mu_imputed                0.087  XXXXX
  clv_imputed               0.078  XXXX
  p_alive_imputed           0.064  XXX
  alpha_imputed             0.052  XXX
  TotalSpend                0.038  XX
  M_Score                   0.036  XX
  behavioral_entropy        0.023  X
  AOV                       0.021  X
  IsUK                      0.018  X
  CSV: 05_feature_importance.csv


## Step 10 - Dashboard visualisations

In [11]:
seg_order  = ["Persuadable", "Sure-Thing", "Sleeping Dog", "Lost Cause"]
seg_colors = {
    "Persuadable":  PALETTE["positive"],
    "Sure-Thing":   PALETTE["primary"],
    "Sleeping Dog": PALETTE["highlight"],
    "Lost Cause":   PALETTE["neutral"],
}

fig = plt.figure(figsize=(18, 14))
fig.suptitle("RQ3: Causal Uplift Framework", fontsize=16, fontweight="bold")

# ── 1: CATE distribution ──────────────────────────────────────────────────────
ax1 = fig.add_subplot(2, 3, 1)
for seg in seg_order:
    vals = model_df.loc[model_df["UpliftSegment"] == seg, "CATE"]
    if len(vals):
        ax1.hist(vals, bins=30, alpha=0.6,
                 label=f"{seg} (n={len(vals)})",
                 color=seg_colors[seg])
ax1.axvline(0, color="black", linestyle="--", lw=1.5)
ax1.set_xlabel("CATE")
ax1.set_title("CATE Distribution by Segment")
ax1.legend(fontsize=8)

# ── 2: Qini curve ─────────────────────────────────────────────────────────────
ax2 = fig.add_subplot(2, 3, 2)
ax2.plot(qini_df["pct_contacted"], qini_df["qini_cate"],
         color=PALETTE["positive"], lw=2.5,
         label=f"CATE-ranked (AUUC={auuc:.4f})")
ax2.plot(qini_df["pct_contacted"], qini_df["qini_random"],
         color=PALETTE["neutral"], lw=1.5, linestyle="--",
         label="Random baseline")
ax2.fill_between(qini_df["pct_contacted"],
                 qini_df["qini_random"], qini_df["qini_cate"],
                 alpha=0.15, color=PALETTE["positive"])
ax2.set_xlabel("Fraction Contacted")
ax2.set_title(f"Qini Curve  AUUC = {auuc:.4f}")
ax2.legend(fontsize=9)

# ── 3: Segment donut ──────────────────────────────────────────────────────────
ax3 = fig.add_subplot(2, 3, 3)
sizes = [seg_counts.get(s, 0) for s in seg_order]
ax3.pie(
    sizes,
    labels=[f"{s}\n{seg_counts.get(s,0):,}" for s in seg_order],
    colors=[seg_colors[s] for s in seg_order],
    autopct="%1.0f%%", startangle=140,
    wedgeprops={"edgecolor": "white", "linewidth": 2},
    textprops={"fontsize": 9},
)
ax3.set_title("Uplift Segment Distribution")

# ── 4: CEO bubble chart ───────────────────────────────────────────────────────
ax4 = fig.add_subplot(2, 3, 4)
plot_df  = model_df[model_df["CounterfactualCLV"] > 0].copy()
bub_size = (
    plot_df["CounterfactualCLV"]
    / plot_df["CounterfactualCLV"].max() * 300
).clip(lower=10)
bub_cols = [seg_colors.get(s, PALETTE["neutral"])
            for s in plot_df["UpliftSegment"]]
ax4.scatter(plot_df["CATE"], plot_df["p_alive_imputed"],
            s=bub_size, c=bub_cols, alpha=0.45)
ax4.axvline(0, color="black", linestyle="--", lw=1)
ax4.set_xlabel("CATE (causal responsiveness)")
ax4.set_ylabel("P(alive) - imputed BG/NBD")
ax4.set_title("CEO Bubble: CATE x P(alive) x CLV")
ax4.legend(
    handles=[mpatches.Patch(color=seg_colors[s], label=s) for s in seg_order],
    fontsize=8, loc="lower right",
)

# ── 5: IPTW covariate balance ─────────────────────────────────────────────────
ax5 = fig.add_subplot(2, 3, 5)
bal_sorted = balance_df.sort_values("smd_raw")
ax5.barh(bal_sorted["feature"], bal_sorted["smd_raw"].abs(),
         label="Before IPTW", color=PALETTE["accent"], alpha=0.6)
ax5.barh(bal_sorted["feature"], bal_sorted["smd_weighted"].abs(),
         label="After IPTW", color=PALETTE["positive"], alpha=0.8)
ax5.axvline(0.1, color="black", linestyle="--", lw=1, label="|SMD| = 0.10 threshold")
ax5.set_xlabel("|Standardised Mean Difference|")
ax5.set_title("IPTW Covariate Balance")
ax5.legend(fontsize=8)

# ── 6: Monte Carlo ROI ────────────────────────────────────────────────────────
ax6 = fig.add_subplot(2, 3, 6)
ax6.fill_between(roi_df["cost_per_contact"],
                 roi_df["roi_p5"] * 100,
                 roi_df["roi_p95"] * 100,
                 alpha=0.25, color=PALETTE["primary"],
                 label="95% CI")
ax6.plot(roi_df["cost_per_contact"], roi_df["mean_roi"] * 100,
         color=PALETTE["primary"], lw=2.5, label="Mean ROI")
ax6.axhline(0, color="black", linestyle="--", lw=1)
ax6.set_xlabel("Cost per contact (GBP)")
ax6.set_ylabel("ROI (%)")
ax6.set_title(f"Monte Carlo ROI  (n=50 Persuadables, {N_ITER:,} iters)")
ax6.legend(fontsize=9)

fig.tight_layout()
path = f"{CHARTS_DIR}chart_05_causal_dashboard.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.close()
print(f"  Saved: {path}")

# Ablation chart
fig2, ax = plt.subplots(figsize=(8, 5))
models  = ["RFM-only\n(baseline)", "RFM + BG/NBD\n(enriched)"]
aucs    = [auc_base, auc_full]
colors  = [PALETTE["neutral"], PALETTE["positive"]]
bars    = ax.bar(models, aucs, color=colors, alpha=0.85, width=0.45)
y_min = min(aucs) * 0.97
y_max = max(aucs) * 1.03
ax.set_ylim(y_min, y_max)
ax.set_ylabel("5-fold CV AUC")
ax.set_title(
    f"Ablation Study: BG/NBD Enrichment Impact\n"
    f"delta AUC = {delta:+.4f}  ({lift_pct:+.1f}%)"
)
for bar, val in zip(bars, aucs):
    ax.text(bar.get_x() + bar.get_width() / 2,
            val + (y_max - y_min) * 0.01,
            f"{val:.4f}", ha="center", fontweight="bold", fontsize=12)
fig2.tight_layout()
path2 = f"{CHARTS_DIR}chart_05_ablation_study.png"
fig2.savefig(path2, dpi=150, bbox_inches="tight")
plt.close()
print(f"  Saved: {path2}")

  Saved: ../outputs/charts/chart_05_causal_dashboard.png
  Saved: ../outputs/charts/chart_05_ablation_study.png


## Save all RQ3 outputs

In [12]:
model_df.to_csv(f"{DATA_DIR}05_uplift_cate.csv", index=False)

pd.DataFrame([
    {
        "model_variant": "Baseline (RFM only)",
        "n_features": len(BASE_FEATURES),
        "cv_auc": auc_base,
        "bgnbd_imputed": False,
        "iptw_applied": True,
    },
    {
        "model_variant": "Enriched (RFM + BG/NBD)",
        "n_features": len(FULL_FEATURES),
        "cv_auc": auc_full,
        "bgnbd_imputed": True,
        "iptw_applied": True,
    },
]).assign(delta_auc=lambda df: df["cv_auc"].diff().fillna(0)).to_csv(
    f"{DATA_DIR}05_ablation_scores.csv", index=False
)

print("=" * 60)
print("  NOTEBOOK 05 SUMMARY")
print("=" * 60)
for seg in seg_order:
    n  = seg_counts.get(seg, 0)
    sp = seg_spend.get(seg, 0.0)
    mc = seg_cate.get(seg, 0.0)
    print(f"  {seg:<16} n={n:>4}  mean CATE={mc:+.3f}  spend=GBP{sp:,.0f}")
print()
print(f"  IPTW balanced  : {n_balanced}/{len(balance_df)} features |SMD|<0.10")
print(f"  AUUC           : {auuc:.4f}")
print(f"  Ablation delta : {delta:+.4f} ({lift_pct:+.1f}%)")
print(f"  Top-50 CLV     : GBP{top50_clv:,.0f}")
print(f"  Sleeping Dogs  : {sd_n}  (GBP{sd_spend:,.0f} spend)")
print(f"  Sample size    : {len(model_df):,} (full population)")

outputs = sorted([f for f in os.listdir(DATA_DIR) if f.startswith("05_")])
print(f"\n  {len(outputs)} CSVs saved:")
for f in outputs:
    kb = os.path.getsize(f"{DATA_DIR}{f}") // 1024
    print(f"    {f:<45} {kb:>5} KB")

  NOTEBOOK 05 SUMMARY
  Persuadable      n= 708  mean CATE=+0.280  spend=GBP1,129,280
  Sure-Thing       n= 290  mean CATE=+0.019  spend=GBP572,831
  Sleeping Dog     n=1380  mean CATE=-0.017  spend=GBP4,883,021
  Lost Cause       n=3474  mean CATE=-0.362  spend=GBP10,844,276

  IPTW balanced  : 10/13 features |SMD|<0.10
  AUUC           : 0.1511
  Ablation delta : +0.0671 (+8.2%)
  Top-50 CLV     : GBP85,855
  Sleeping Dogs  : 1380  (GBP4,883,021 spend)
  Sample size    : 5,852 (full population)

  9 CSVs saved:
    05_ablation_scores.csv                            0 KB
    05_feature_importance.csv                         0 KB
    05_intervention_priority.csv                      4 KB
    05_iptw_balance.csv                               1 KB
    05_monte_carlo_roi.csv                            1 KB
    05_propensity_scores.csv                        259 KB
    05_qini_curve.csv                               354 KB
    05_sleeping_dog_blacklist.csv                    64 KB
    05_up